# Arbres — CO₂, masse, puissance et type de boîte des véhicules

*Machine Learning Specialization (DeepLearning.AI, cours 2 — semaine 4), appliquée au jeu de données de l'ADEME sur les émissions des véhicules commercialisés en France.*

À la fin du deuxième notebook, j'avançais que les erreurs de prédiction sur le haut du parc (les véhicules les plus puissants) venaient de la carrosserie et de l'aérodynamique, deux informations absentes du fichier. Ce notebook met cette hypothèse à l'épreuve : si des modèles capables de capter les non-linéarités et les interactions réduisent nettement ces erreurs, c'est que le modèle linéaire était en cause, et non les variables manquantes.

**Plan**

0. Références
1. Un arbre seul (`DecisionTreeRegressor`)
2. La forêt aléatoire (`RandomForestRegressor`)
3. Le boosting (`HistGradientBoostingRegressor` et `XGBRegressor`)
4. Comparaison appariée
5. Où se fait le gain
6. Ce que les arbres ne savent pas faire
7. Conclusion et choix de l'API

## Partie 0 - Références

In [1]:
import warnings
warnings.filterwarnings("ignore", message="Found unknown categories", category=UserWarning)

import numpy as np
import pandas as pd

from sklearn.base import clone, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor

from prepa_arbres_boosting import charger, preparer

CIBLE = "co2_mixte"
CARACTERISTIQUES = ["masse_ordma_min", "puiss_max", "typ_boite_nb_rapp"]

TEST_SIZE = 0.2
RANDOM_STATE = 0

In [2]:
# pipeline pour la régression linéaire
def construire_pipe_lineaire(estimateur: RegressorMixin) -> Pipeline:
    preparation = ColumnTransformer(
        [
            ("masse", StandardScaler(), ["masse_ordma_min"]),
            ("puissance",
             make_pipeline(PolynomialFeatures(degree=2, include_bias=False), StandardScaler()),
             ["puiss_max"]),
            ("boite",
             OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False),
             ["typ_boite_nb_rapp"]),
        ]
    )
    return Pipeline([("preparation", preparation), ("estimateur", clone(estimateur))])

In [3]:
# chargement et préparation des données
df = charger("data/vehicules.csv")
d = preparer(df, 'ES')


# découpage aléatoire, utilisé seulement pour mesurer les doublons
X, y = d[CARACTERISTIQUES], d[CIBLE]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

**Vérifier l'ampleur des doublons**

In [4]:
cle = lambda X: X[CARACTERISTIQUES].astype(str).agg("|".join, axis=1)

cles_tr, cles_te = cle(X_tr), cle(X_te)
print(f"{cles_tr.nunique()} configurations distinctes pour {len(cles_tr)} véhicules (entraînement)")
print(f"{cles_te.isin(cles_tr).mean():.0%} des véhicules du test ont un jumeau à l'entraînement")

1240 configurations distinctes pour 1924 véhicules (entraînement)
56% des véhicules du test ont un jumeau à l'entraînement


J'ai ajouté cette vérification après un résultat suspect avec le `DecisionTreeRegressor`. Le meilleur arbre était celui avec `max_depth=None` et `min_samples_leaf=1`, c'est-à-dire un arbre qui se développe jusqu'au bout : le plus complexe possible. Il apprenait presque les données par cœur, avec une MAE de 1,6 g/km à l'entraînement contre 9,1 g/km en validation.

La cause la plus probable venait des données elles-mêmes, et les chiffres le confirment. On compte 1 924 véhicules pour seulement 1 240 configurations distinctes, soit environ 680 lignes qui répètent une configuration déjà présente. Surtout, plus de la moitié des véhicules évalués (56 %) ont un « jumeau » dans les données d'entraînement. Avec un découpage aléatoire, apprendre par cœur est donc récompensé. C'est pour cela que l'arbre le plus profond l'emportait, et c'est aussi pour cela que l'erreur de validation est trop optimiste.

Pour la suite, j'ai donc choisi de découper les données par configuration, avec `GroupKFold` pour la validation croisée et `GroupShuffleSplit` pour la séparation entre entraînement et test. Ainsi, une même configuration ne se retrouve jamais à la fois dans les données d'entraînement et dans celles d'évaluation.

In [5]:
# séparation les données entre entraînement et test
groupes = cle(X)
idx_tr, idx_te = next(
    GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    .split(X, y, groups=groupes)
)
X_tr, X_te, y_tr, y_te = X.iloc[idx_tr], X.iloc[idx_te], y.iloc[idx_tr], y.iloc[idx_te]
groupes_tr = groupes.iloc[idx_tr]


# validation croisée : mêmes plis pour tous les modèles
plis = GroupKFold(n_splits=5)

In [6]:
# gardés pour la comparaison appariée (partie 4)
scores_par_pli = {}

# fonction d'évaluation des modèles
def evaluer(nom: str, modele: Pipeline) -> pd.Series:
    r = cross_validate(
        modele, X_tr, y_tr, cv=plis, groups=groupes_tr,
        scoring="neg_mean_absolute_error", return_train_score=True,
    )

    scores_par_pli[nom] = -r['test_score']

    return pd.Series(
        {
            "MAE validation (g/km)": -r['test_score'].mean(),
            "écart-type (g/km)": r['test_score'].std(),
            "MAE entraînement (g/km)": -r['train_score'].mean(),
        },
        name=nom,
    )

La régression de `co2-api` et le modèle naïf suivant évalués avec ce protocole sont les barres à battre.

In [7]:
resultats = pd.DataFrame([
    evaluer("naïf (moyenne)", construire_pipe_lineaire(DummyRegressor())),
    evaluer("régression co2-api", construire_pipe_lineaire(LinearRegression())),
])
resultats

,MAE validation (g/km),écart-type (g/km),MAE entraînement (g/km)
naïf (moyenne),42.707628,1.487277,42.694531
régression co2-api,16.134781,0.913420,15.667945


In [8]:
X_tr["typ_boite_nb_rapp"].value_counts()

typ_boite_nb_rapp
A 7    562
M 6    546
M 5    247
A 8    209
A 6    192
V 0     29
A 5     26
D 7     19
D 5     18
M 7     15
A 4     13
A 9     13
D 6      5
A 0      2
Name: count, dtype: int64

Le découpage par configuration place parfois les modalités très rares (D 6 : 5 véhicules, A 0 : 2) uniquement dans un pli de validation, d’où l’avertissement de OneHotEncoder. Cela concerne moins de 0,5 % des véhicules ; l’effet sur la MAE est négligeable et je garde le pipeline de co2-api tel quel pour que la référence reste fidèle.

## Partie 1 - Un seul arbre  (`DecisionTreeRegressor`)

In [9]:
# Construire le pipeline pour les arbres
def construire_pipe_arbre(estimateur: RegressorMixin) -> Pipeline:
    preparation = ColumnTransformer(
        [
            ("num", "passthrough", ["masse_ordma_min", "puiss_max"]),
            (
                "boite",
                OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                ["typ_boite_nb_rapp"],
            ),
        ]
    )
    return Pipeline([("preparation", preparation), ("estimateur", clone(estimateur))])

In [10]:
# hyperparamètres pour l'arbre
grille_arbre = {
    "estimateur__max_depth": [3, 5, 8, 12, None],
    "estimateur__min_samples_leaf": [1, 5, 10, 20],
}

In [11]:
# Entraînement et évaluation du modèle DecisionTreeRegressor 
recherche = GridSearchCV(
    construire_pipe_arbre(DecisionTreeRegressor(random_state=RANDOM_STATE)),
    grille_arbre, cv=plis, scoring="neg_mean_absolute_error", return_train_score=True,
)
recherche.fit(X_tr, y_tr, groups=groupes_tr)
print(recherche.best_params_, -recherche.best_score_)

resultats.loc["arbre réglé"] = evaluer("arbre réglé", recherche.best_estimator_)
resultats

{'estimateur__max_depth': 12, 'estimateur__min_samples_leaf': 1} 12.68918946192975


,MAE validation (g/km),écart-type (g/km),MAE entraînement (g/km)
naïf (moyenne),42.707628,1.487277,42.694531
régression co2-api,16.134781,0.913420,15.667945
arbre réglé,12.689189,0.553003,4.236297


Avec le découpage par configuration, l’arbre passe de 9,1 à 12,7 g/km quand la régression passe seulement de 15,6 à 16,1.